# 01 — Fuente de Datos: Chicago Crimes — Análisis COVID 2017–2025

## Origen

El dataset proviene del **Chicago Data Portal** ([data.cityofchicago.org](https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2)), el portal oficial de datos abiertos de la ciudad de Chicago, administrado por el **Chicago Police Department (CPD)**.

El CPD registra cada incidente criminal reportado en la ciudad desde el año 2001, con actualización continua. Los datos son de acceso público bajo licencia **Open Data Commons Attribution License**, lo que permite su uso sin restricciones para análisis académico y comercial.

---

## Obtención de los Datos

### API Pública SODA (Socrata Open Data API)

El Chicago Data Portal expone los datos a través de la **SODA API** de Socrata, el estándar de facto para portales de datos abiertos municipales en EE.UU. El endpoint del dataset es:

```
https://data.cityofchicago.org/resource/ijzp-q8t2.json
```

| Método | Descripción |
|---|---|
| `GET /resource/ijzp-q8t2.json` | Retorna registros en formato JSON (máx. 1,000 por llamada) |
| `GET /resource/ijzp-q8t2.csv` | Retorna registros en formato CSV |
| Parámetro `$limit` | Controla el número de filas devueltas |
| Parámetro `$where` | Filtros tipo SQL (`year=2020`, `arrest=true`) |
| Parámetro `$order` | Ordenación de resultados |

Para volúmenes mayores a 50,000 registros, el portal ofrece **exportación directa** por año en formato CSV — que es el método que utilizamos en este proyecto para garantizar la descarga completa y reproducible de todos los registros históricos.

### Pipeline de Obtención y Preparación

```
Chicago Data Portal          Parseo.ipynb              Google Cloud Storage
(CSV por año 2001–2026)  ──►  Normalización  ──►  gs://big-data-proyecto-parcial/raw/
     26 archivos              de 3 formatos          26 CSVs listos para Dask/Spark
```

### Normalización con Parseo.ipynb

El dataset histórico de Chicago Crimes existe en **3 formatos distintos de CSV** debido a cambios en el sistema de registro del CPD a lo largo de los años. `Parseo.ipynb` detecta automáticamente el formato de cada archivo y los unifica bajo un único schema:

| Aspecto | Formato A (años recientes) | Formato B/C (años anteriores) |
|---|---|---|
| **Nombres de columnas** | `snake_case` (`unique_key`, `primary_type`) | `Title Case` (`ID`, `Primary Type`) |
| **Formato de fecha** | ISO 8601 UTC (`2020-03-29 03:45:00+00:00`) | 12h AM/PM (`03/29/2020 03:45:00 AM`) |
| **Formato `updated_on`** | ISO 8601 UTC | `2018 Feb 10 03:50:01 PM` |
| **Columna `location`** | `(lat, lon)` — coordenadas entre paréntesis | WKT `POINT (lon lat)` |

**Operaciones de normalización aplicadas por `Parseo.ipynb`:**

| Operación | Detalle |
|---|---|
| Renombre de columnas | Formatos B/C → `snake_case` vía diccionario `COLUMN_MAP` |
| Conversión de fechas | Detecta formato automáticamente → convierte a `TIMESTAMP UTC` |
| Normalización de booleanos | `"True"/"False"` string → `bool` nativo |
| Conversión a enteros nullable | `Int64` de pandas — tolera NaN en `district`, `ward`, `community_area` |
| Estandarización de `location` | `(lat, lon)` → WKT `POINT (lon lat)` para compatibilidad GIS |
| Reemplazo de vacíos | Strings vacíos `""` → `NaN` en todas las columnas |

Esta normalización es crítica para el procesamiento distribuido: Dask, Modin y Spark requieren schemas consistentes para leer múltiples archivos como un único DataFrame sin errores de tipo.

---

## Enfoque del Análisis: Impacto del COVID-19 en la Criminalidad de Chicago

Este proyecto estudia cómo la pandemia de COVID-19 transformó los patrones de criminalidad en Chicago, comparando tres eras definidas en función de los eventos epidemiológicos y las políticas públicas asociadas:

| Era | Años | Contexto |
|---|---|---|
| **PRE-COVID** | 2017, 2018, 2019 | Normalidad pre-pandémica. Línea base de referencia. |
| **DURANTE COVID** | 2020, 2021, 2022 | Pandemia activa: confinamiento (marzo 2020), reaperturas graduales, variantes Delta y Ómicron. |
| **POST-COVID** | 2023, 2024, 2025 | Recuperación: levantamiento de restricciones, reactivación económica, nuevo equilibrio social. |

### ¿Por qué este período?

El COVID-19 fue el evento social más disruptivo del siglo XXI. En Chicago:
- **Marzo 2020:** Gobernador declara estado de emergencia; cierre de negocios, toque de queda.
- **2020:** Los crímenes callejeros cayeron, pero la violencia doméstica aumentó por el confinamiento.
- **2021:** Reapertura gradual con repuntes de violencia en zonas históricamente vulnerables.
- **2022:** Normalización parcial con cambios estructurales en patrones de movilidad.
- **2023–2025:** Retorno a tendencias pre-pandémicas con diferencias notables en algunas categorías.

---

## Estructura del Dataset Analizado

| Propiedad | Valor |
|---|---|
| **Período analizado** | 2017–2025 (9 años) |
| **Archivos utilizados** | 9 CSVs (uno por año) |
| **Total de registros** | ~2.07 millones de incidentes |
| **Columnas** | 22 variables + `covid_era` (columna derivada) |
| **Formato** | CSV, separado por comas, encoding UTF-8 |
| **Dataset completo disponible** | 2001–2026 en GCS (`gs://big-data-proyecto-parcial/raw/`) |

---

## Variables Principales

### Identificadores del incidente

| Variable | Tipo | Contexto real |
|---|---|---|
| `unique_key` | INT64 | Identificador numérico único asignado por el sistema CPD a cada incidente. Sirve como clave primaria en el dataset. |
| `case_number` | STRING | Número de caso oficial policial (formato: letra + 6 dígitos, ej. `HZ451234`). Es el identificador utilizado en investigaciones y juicios. |

### Temporalidad

| Variable | Tipo | Contexto real |
|---|---|---|
| `date` | TIMESTAMP (UTC) | Fecha y hora exacta en que ocurrió el incidente según el reporte policial. |
| `year` | INT64 | Año del incidente. Columna de particionamiento. |
| `covid_era` | STRING (derivada) | Clasificación de la era: `PRE` (2017–2019), `DURANTE` (2020–2022), `POST` (2023–2025). |
| `updated_on` | TIMESTAMP (UTC) | Fecha de la última modificación del registro en el sistema CPD. |

### Ubicación

| Variable | Tipo | Contexto real |
|---|---|---|
| `block` | STRING | Bloque anonimizado donde ocurrió el incidente (ej. `001XX W MADISON ST`). |
| `beat` | INT64 | Unidad de patrullaje más pequeña del CPD (~300 beats en Chicago). |
| `district` | INT64 | Distrito policial (1–25). Cada distrito tiene su propia comisaría. |
| `ward` | INT64 | Distrito político/aldermánico (1–50). |
| `community_area` | INT64 | Área comunitaria oficial (1–77, ej. 25=Austin, 8=Near North Side, 32=Loop). |
| `latitude` / `longitude` | FLOAT64 | Coordenadas WGS84, anonimizadas al centroide del bloque por protección de víctimas. |
| `location` | STRING | WKT: `POINT (longitud latitud)`. Para análisis GIS y mapas en Power BI. |

### Clasificación del crimen

| Variable | Tipo | Contexto real |
|---|---|---|
| `primary_type` | STRING | Categoría principal del crimen (THEFT, BATTERY, CRIMINAL DAMAGE, NARCOTICS, ASSAULT, etc.). Uniformizada contra el catálogo oficial IUCR. |
| `description` | STRING | Subcategoría específica dentro del `primary_type`. |
| `iucr` | STRING | Código Illinois Uniform Crime Reporting — clasificación estatal del delito. |
| `fbi_code` | STRING | Clasificación federal FBI UCR. Part I (01A–09): crímenes graves. Part II (10–26): crímenes menores. |
| `location_description` | STRING | Tipo de lugar físico (~170 categorías: STREET, RESIDENCE, CTA BUS, SCHOOL, etc.). |

### Resultado del incidente

| Variable | Tipo | Contexto real |
|---|---|---|
| `arrest` | BOOL | `True` si el incidente resultó en un arresto formal. |
| `domestic` | BOOL | `True` si fue clasificado como violencia doméstica (Illinois Domestic Violence Act). Especialmente relevante durante el confinamiento de 2020. |

---

## Calidad del Dataset

| Aspecto | Detalle |
|---|---|
| **IUCR uniformizado** | `primary_type` y `description` corregidas contra el catálogo oficial IUCR.csv — elimina abreviaciones históricas inconsistentes. |
| **Registros sin coordenadas** | ~1-3% carecen de lat/lon. Se mantienen para análisis no-geoespaciales. |
| **2025 incompleto** | El año 2025 tiene datos hasta la fecha de extracción — se indica en visualizaciones. |
| **Crímenes no reportados** | Solo incluye crímenes reportados al CPD — sesgo inherente en todos los datasets policiales. |

In [1]:
import pandas as pd
import os

FOLDER      = 'Chicago_Crimes_by_Year'
YEARS       = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ERA_MAP     = {y: ('PRE' if y <= 2019 else ('DURANTE' if y <= 2022 else 'POST')) for y in YEARS}
ERA_LABELS  = {'PRE': '2017–2019', 'DURANTE': '2020–2022', 'POST': '2023–2025'}

print('Dataset analizado: Chicago Crimes — Impacto COVID-19')
print('─' * 62)
print(f'  {"Archivo":<35} {"Era":<8} {"Filas":>10} {"MB":>7}')
print('─' * 62)

total_rows = 0
total_mb   = 0
era_counts = {'PRE': 0, 'DURANTE': 0, 'POST': 0}

for y in YEARS:
    fname = f'Chicago_Crimes_{y}.csv'
    path  = os.path.join(FOLDER, fname)
    nrows = sum(1 for _ in open(path, encoding='utf-8')) - 1
    mb    = os.path.getsize(path) / (1024**2)
    era   = ERA_MAP[y]
    era_counts[era] += nrows
    total_rows += nrows
    total_mb   += mb
    print(f'  {fname:<35} {era:<8} {nrows:>10,} {mb:>7.1f}')

print('─' * 62)
print(f'  {"TOTAL":<35} {"":8} {total_rows:>10,} {total_mb:>7.1f}')
print()
print('Registros por era:')
for era, label in ERA_LABELS.items():
    pct = era_counts[era] / total_rows * 100
    print(f'  {era:<8} ({label}): {era_counts[era]:>9,}  ({pct:.1f}%)')

Dataset analizado: Chicago Crimes — Impacto COVID-19
──────────────────────────────────────────────────────────────
  Archivo                             Era           Filas      MB
──────────────────────────────────────────────────────────────
  Chicago_Crimes_2017.csv             PRE         269,214    63.1


  Chicago_Crimes_2018.csv             PRE         269,070    63.1
  Chicago_Crimes_2019.csv             PRE         261,555    61.6


  Chicago_Crimes_2020.csv             DURANTE     212,522    50.1
  Chicago_Crimes_2021.csv             DURANTE     209,406    49.2


  Chicago_Crimes_2022.csv             DURANTE     239,655    56.2
  Chicago_Crimes_2023.csv             POST        262,756    61.9


  Chicago_Crimes_2024.csv             POST        256,305    60.4
  Chicago_Crimes_2025.csv             POST         92,460    21.9
──────────────────────────────────────────────────────────────
  TOTAL                                         2,072,943   487.4

Registros por era:
  PRE      (2017–2019):   799,839  (38.6%)
  DURANTE  (2020–2022):   661,583  (31.9%)
  POST     (2023–2025):   611,521  (29.5%)


In [2]:
sample = pd.read_csv(os.path.join(FOLDER, 'Chicago_Crimes_2020.csv'), nrows=3)
print('Columnas y tipos de datos:')
print(sample.dtypes.to_string())
print()
print('Muestra de 3 registros (año 2020 — inicio del COVID):')
sample

Columnas y tipos de datos:
unique_key                int64
case_number              object
date                     object
block                    object
iucr                      int64
primary_type             object
description              object
location_description     object
arrest                     bool
domestic                   bool
beat                      int64
district                  int64
ward                      int64
community_area            int64
fbi_code                  int64
x_coordinate              int64
y_coordinate              int64
year                      int64
updated_on               object
latitude                float64
longitude               float64
location                 object

Muestra de 3 registros (año 2020 — inicio del COVID):


,unique_key,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
0,12019581,JD197307,2020-03-29 03:45:00+00:00,0000X E WACKER DR,334,ROBBERY,ATTEMPT ARMED - KNIFE / CUTTING INSTRUMENT,CONVENIENCE STORE,False,False,...,42,32,3,1176621,1902155,2020,2020-04-05 15:41:51+00:00,41.886864,-87.626852,POINT (-87.626851797 41.886863814)
1,11940387,JD102775,2020-01-01 00:30:00+00:00,002XX N STATE ST,870,THEFT,POCKET-PICKING,SIDEWALK,False,False,...,42,32,6,1176327,1901797,2020,2020-01-08 15:46:16+00:00,41.885888,-87.627942,POINT (-87.627942238 41.885888079)
2,11980069,JD144543,2020-02-08 11:00:00+00:00,0000X W RANDOLPH ST,870,THEFT,POCKET-PICKING,RESTAURANT,False,False,...,42,32,6,1175768,1901280,2020,2020-02-19 15:39:26+00:00,41.884482,-87.630011,POINT (-87.630010542 41.884481993)


In [3]:
# Tipos de crimen disponibles en el período 2017–2025
frames = [pd.read_csv(os.path.join(FOLDER, f'Chicago_Crimes_{y}.csv'),
                      usecols=['primary_type'], dtype=str) for y in YEARS]
all_types = pd.concat(frames)['primary_type'].value_counts().reset_index()
all_types.columns = ['primary_type', 'total_2017_2025']
print(f'Tipos de crimen únicos en el período analizado: {len(all_types)}')
print(all_types.to_string(index=False))

Tipos de crimen únicos en el período analizado: 33
                     primary_type  total_2017_2025
                            THEFT           467705
                          BATTERY           378849
                  CRIMINAL DAMAGE           229523
                          ASSAULT           174527
               DECEPTIVE PRACTICE           148781
                    OTHER OFFENSE           131946
              MOTOR VEHICLE THEFT           129867
                          ROBBERY            77030
                         BURGLARY            76546
                        NARCOTICS            73048
                WEAPONS VIOLATION            61505
                CRIMINAL TRESPASS            44139
       OFFENSE INVOLVING CHILDREN            17231
          CRIMINAL SEXUAL ASSAULT            13263
                      SEX OFFENSE            10054
           PUBLIC PEACE VIOLATION             9226
 INTERFERENCE WITH PUBLIC OFFICER             6971
                         HOMICI